In [0]:
from pyspark.sql import functions as F

# 1. Processamento Silver para o VRA (Voos)
df_vra_bronze = spark.read.table("voe_bem.bronze.vra")

df_vra_silver = (
    df_vra_bronze
    .withColumn("Partida_Prevista", F.coalesce(F.try_to_timestamp("Partida_Prevista", F.lit("yyyy-MM-dd HH:mm:ss")), F.try_to_timestamp("Partida_Prevista", F.lit("yyyy-MM-dd HH:mm:ss.SSSSSSSSS"))))
    .withColumn("Partida_Real", F.coalesce(F.try_to_timestamp("Partida_Real", F.lit("yyyy-MM-dd HH:mm:ss")), F.try_to_timestamp("Partida_Real", F.lit("yyyy-MM-dd HH:mm:ss.SSSSSSSSS"))))
    .withColumn("Chegada_Prevista", F.coalesce(F.try_to_timestamp("Chegada_Prevista", F.lit("yyyy-MM-dd HH:mm:ss")), F.try_to_timestamp("Chegada_Prevista", F.lit("yyyy-MM-dd HH:mm:ss.SSSSSSSSS"))))
    .withColumn("Chegada_Real", F.coalesce(F.try_to_timestamp("Chegada_Real", F.lit("yyyy-MM-dd HH:mm:ss")), F.try_to_timestamp("Chegada_Real", F.lit("yyyy-MM-dd HH:mm:ss.SSSSSSSSS"))))
    .dropDuplicates()
)

# Salvando na tabela Delta Silver
df_vra_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("voe_bem.silver.vra")
print("Tabela silver.vra salva com sucesso!")


# 2. Processamento Silver para Aeródromos
df_aerodromos_bronze = spark.read.table("voe_bem.bronze.aerodromos")

df_aerodromos_silver = (
    df_aerodromos_bronze
    .dropDuplicates()
    # Adicione limpezas específicas se necessário (ex: trim em strings)
    .withColumn("OACI", F.trim(F.col("OACI"))) if "OACI" in df_aerodromos_bronze.columns else df_aerodromos_bronze
)

df_aerodromos_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("voe_bem.silver.aerodromos")
print("Tabela silver.aerodromos salva com sucesso!")